In [4]:
# !pip install --upgrade pip

# # ember github for reference 
!git clone https://github.com/FutureComputing4AI/EMBER2024.git
%pip install ./EMBER2024

%pip install pandas
%pip install altair

# # thrember dependencies
!pip uninstall -y signify
%pip install "signify==0.7.1"

# might need libomp installed:
!brew install libomp

fatal: destination path 'EMBER2024' already exists and is not an empty directory.
Processing ./EMBER2024
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for thrember: filename=thrember-0.1.0-py3-none-any.whl size=30669 sha256=e521aa8df6f02be72f8f3a3110b0afc79e2122df05f1343f3e7d85838dd04e66
  Stored in directory: /private/var/folders/s0/hh3zvcw56y7bxdk45kxvs7nr0000gn/T/pip-ephem-wheel-cache-jg58mp2g/wheels/86/6d/51/314f512104513b1ffffb3c8be7efa7034e55e0af206958531d
Failed to build thrember

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> thrember
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install -

In [5]:
# imports
import os
from pathlib import Path
import thrember
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import lightgbm as lgb
import polars as pl
import altair as alt
from sklearn.metrics import roc_auc_score, roc_curve
alt.renderers.enable('default')

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RendererRegistry.enable('default')

In [6]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"
train_df = pd.read_parquet(DATA_DIR / "win32_detection_train_20pct.parquet")

print("rows:", len(train_df))
print("duplicate rows:", train_df.duplicated().sum())
print("duplicate sha256:", train_df["sha256"].duplicated().sum())


print("Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

display(train_df.head())


print("\nData types:")
print(train_df.dtypes)

print(train_df.iloc[2].strings)
print(train_df.iloc[2].general)



rows: 312125
duplicate rows: 0
duplicate sha256: 0
Shape: (312125, 5)

Columns:
['sha256', 'label', 'general', 'strings', 'imports']


,sha256,label,general,strings,imports
0,000742aa22166e00603ab1101adfd75c793f6ff38cc3f9...,0,"{""size"": 4579380, ""entropy"": 7.98444658339403,...","{""numstrings"": 22744, ""avlength"": 5.8437829757...","{""KERNEL32.dll"": [""GetDriveTypeA"", ""GetModuleF..."
1,000adea549ab7604629f3b606bbcf2afd5aec6f7ed0e6c...,0,"{""size"": 152576, ""entropy"": 3.710459087578284,...","{""numstrings"": 19, ""avlength"": 24.473684210526...",{}
2,0015cbb011bbeb891b3ea505a6995d17395c4d4fd144b0...,1,"{""size"": 122880, ""entropy"": 7.2352091959923825...","{""numstrings"": 838, ""avlength"": 33.65632458233...","{""wsock32.dll"": [""WSAGetLastError"", ""WSAStartu..."
3,001ab6c768246a86b9c784630fda6574980980da5e11bd...,0,"{""size"": 34392, ""entropy"": 6.492246788417963, ...","{""numstrings"": 291, ""avlength"": 14.57731958762...","{""ADVAPI32.dll"": [""RegCloseKey"", ""RegCreateKey..."
4,0020532d7497a685cf1d75668eef3184a0e683fd620090...,1,"{""size"": 437224, ""entropy"": 6.436412852928497,...","{""numstrings"": 9960, ""avlength"": 10.5661646586...","{""MFC42.DLL"": [""MFC42.DLL:ordinal2770"", ""MFC42..."



Data types:
sha256       str
label      int64
general      str
strings      str
imports      str
dtype: object
{"numstrings": 838, "avlength": 33.656324582338904, "printabledist": [3618, 37, 16, 33, 16, 74, 8, 29, 49, 58, 13, 14, 194, 13, 213, 53, 21, 39, 34, 28, 16, 15, 2, 12, 17, 10, 56, 25, 2, 5, 10, 11, 18, 185, 53, 109, 108, 151, 81, 71, 49, 232, 27, 33, 132, 254, 73, 165, 92, 17, 109, 140, 167, 98, 28, 57, 25, 55, 3, 11, 9, 19, 2, 32, 72, 1671, 223, 784, 845, 2722, 493, 377, 584, 1480, 53, 140, 832, 548, 1427, 1694, 608, 33, 1574, 1285, 1731, 700, 202, 233, 67, 419, 29, 5, 9, 1, 5, 12], "printables": 28204, "entropy": 4.89426326751709, "string_counts": {"cache": 3, "clipboard": 2, "command": 1, "connect": 5, "cookie": 1, "create": 14, "crypt": 1, "delete": 4, "desktop": 5, "directory": 4, "dos_msg": 2, "download": 4, "enum": 1, "environment": 1, "exit": 4, "file": 18, "html": 1, "http://": 2, "install": 15, "internet": 6, "memory": 1, "module": 3, "mutex": 2, "password": 1, "pro

In [7]:
label_counts = (
    train_df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

label_counts["category"] = label_counts["label"].map({
    0: "Benign",
    1: "Malware"
})

chart = alt.Chart(label_counts).mark_bar().encode(
    x=alt.X("category:N", title="File Classification"),
    y=alt.Y("count:Q", title="Number of Samples"),
    tooltip=["category", "count"]
).properties(
    title="Distribution of Malware and Benign Samples",
    width=500,
    height=350
)





chart

alt.Chart(...)

In [8]:
#First inspect one imports value:
import json

sample_imports = train_df.iloc[0]["imports"]

if isinstance(sample_imports, str):
    sample_imports = json.loads(sample_imports)

print(type(sample_imports))
print(sample_imports)

<class 'dict'>
{'KERNEL32.dll': ['GetDriveTypeA', 'GetModuleFileNameA', 'GetVersionExA', 'GetVersion', 'CompareStringA', 'GetTimeZoneInformation', 'IsBadCodePtr', 'IsBadReadPtr', 'SetUnhandledExceptionFilter', 'GetStringTypeW', 'GetStringTypeA', 'GetFileType', 'GetStdHandle', 'SetHandleCount', 'GetEnvironmentStringsW', 'GetEnvironmentStrings', 'FreeEnvironmentStringsW', 'FreeEnvironmentStringsA', 'UnhandledExceptionFilter', 'GetOEMCP', 'GetACP', 'GetCPInfo', 'LCMapStringW', 'LCMapStringA', 'GetCurrentProcess', 'HeapReAlloc', 'VirtualAlloc', 'VirtualFree', 'HeapCreate', 'HeapDestroy', 'GetEnvironmentVariableA', 'GetCommandLineA', 'GetStartupInfoA', 'FileTimeToLocalFileTime', 'FileTimeToSystemTime', 'FindNextFileA', 'RemoveDirectoryA', 'MoveFileA', 'RtlUnwind', 'DeleteFileA', 'SetEnvironmentVariableA', 'CreateDirectoryA', 'HeapFree', 'HeapAlloc', 'HeapCompact', 'TerminateProcess', 'ExitProcess', 'GetFileAttributesA', 'SetFileAttributesA', 'MoveFileExA', 'GetModuleHandleA', 'FormatMessage

In [9]:
#Number of imported APIs per file
import json
import pandas as pd
import altair as alt

def parse_imports(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}


def count_imported_apis(import_data):
    if isinstance(import_data, dict):
        total = 0

        for apis in import_data.values():
            if isinstance(apis, list):
                total += len(apis)

        return total

    if isinstance(import_data, list):
        return len(import_data)

    return 0


parsed_imports = train_df["imports"].apply(parse_imports)

api_count_df = pd.DataFrame({
    "api_count": parsed_imports.apply(count_imported_apis),
    "label": train_df["label"]
})

api_count_df["category"] = api_count_df["label"].map({
    0: "Benign",
    1: "Malware"
})

display(
    api_count_df.groupby("category")["api_count"]
    .agg(["count", "mean", "median", "std", "max"])
    .round(2)
)

,count,mean,median,std,max
category,,,,,
Benign,155768,202.50,117.0,348.65,8192
Malware,156357,106.83,84.0,127.61,4616


In [10]:
#Because the dataset is large, aggregate the histogram before sending it to Altair:
import numpy as np

upper_limit = api_count_df["api_count"].quantile(0.99)

api_plot_df = api_count_df[
    api_count_df["api_count"] <= upper_limit
].copy()

bin_edges = np.linspace(
    api_plot_df["api_count"].min(),
    api_plot_df["api_count"].max(),
    41
)

api_plot_df["api_bin"] = pd.cut(
    api_plot_df["api_count"],
    bins=bin_edges,
    include_lowest=True
)

api_histogram_df = (
    api_plot_df
    .groupby(["category", "api_bin"], observed=True)
    .size()
    .reset_index(name="count")
)

api_histogram_df["bin_midpoint"] = (
    api_histogram_df["api_bin"]
    .apply(lambda interval: interval.mid)
    .astype(float)
)

api_histogram_df["percentage"] = (
    api_histogram_df.groupby("category")["count"]
    .transform(lambda values: values / values.sum() * 100)
)

api_histogram_plot = api_histogram_df[
    ["category", "bin_midpoint", "count", "percentage"]
].copy()

In [11]:
#Plot it:
alt.Chart(api_histogram_plot).mark_line(
    point=True
).encode(
    x=alt.X(
        "bin_midpoint:Q",
        title="Number of imported APIs"
    ),
    y=alt.Y(
        "percentage:Q",
        title="Percentage of files"
    ),
    color=alt.Color(
        "category:N",
        title="File type"
    ),
    tooltip=[
        "category:N",
        alt.Tooltip(
            "bin_midpoint:Q",
            title="Imported APIs",
            format=".0f"
        ),
        alt.Tooltip(
            "percentage:Q",
            title="Percentage",
            format=".2f"
        )
    ]
).properties(
    title="Imported API Count: Malware vs. Benign",
    width=650,
    height=400
)

alt.Chart(...)

In [12]:
"""Flatten the imported API names

This converts each file’s nested imports into one list:"""
def flatten_imported_apis(import_data):
    api_names = []

    if isinstance(import_data, dict):
        for apis in import_data.values():
            if isinstance(apis, list):
                api_names.extend(
                    str(api).lower()
                    for api in apis
                )

    elif isinstance(import_data, list):
        api_names.extend(
            str(api).lower()
            for api in import_data
        )

    return api_names


api_lists = parsed_imports.apply(flatten_imported_apis)

print(api_lists.iloc[0][:20])

['getdrivetypea', 'getmodulefilenamea', 'getversionexa', 'getversion', 'comparestringa', 'gettimezoneinformation', 'isbadcodeptr', 'isbadreadptr', 'setunhandledexceptionfilter', 'getstringtypew', 'getstringtypea', 'getfiletype', 'getstdhandle', 'sethandlecount', 'getenvironmentstringsw', 'getenvironmentstrings', 'freeenvironmentstringsw', 'freeenvironmentstringsa', 'unhandledexceptionfilter', 'getoemcp']


In [13]:
"""Compare networking API usage

Define a reasonable set of networking-related terms:"""

network_terms = [
    "socket",
    "connect",
    "send",
    "recv",
    "bind",
    "listen",
    "accept",
    "internetopen",
    "internetconnect",
    "internetreadfile",
    "internetwritefile",
    "httpopenrequest",
    "httpsendrequest",
    "urlopen",
    "urldownloadtofile",
    "winhttpopen",
    "winhttpconnect",
    "winhttpsendrequest",
    "wsastartup"
]

In [14]:
#Count networking APIs per file:
def count_matching_apis(api_names, search_terms):
    return sum(
        any(term in api_name for term in search_terms)
        for api_name in api_names
    )


network_df = pd.DataFrame({
    "network_api_count": api_lists.apply(
        lambda names: count_matching_apis(names, network_terms)
    ),
    "label": train_df["label"]
})

network_df["category"] = network_df["label"].map({
    0: "Benign",
    1: "Malware"
})

In [15]:
#Calculate how many files import at least one networking API:
network_summary = (
    network_df.assign(
        uses_network_api=network_df["network_api_count"] > 0
    )
    .groupby("category")
    .agg(
        files=("uses_network_api", "size"),
        files_using_networking=("uses_network_api", "sum"),
        average_network_apis=("network_api_count", "mean"),
        median_network_apis=("network_api_count", "median")
    )
    .reset_index()
)

network_summary["percentage_using_networking"] = (
    network_summary["files_using_networking"]
    / network_summary["files"]
    * 100
)

display(network_summary.round(2))

,category,files,files_using_networking,average_network_apis,median_network_apis,percentage_using_networking
0,Benign,155768,63518,2.67,0.0,40.78
1,Malware,156357,72199,2.29,0.0,46.18


In [16]:
#Visualize the percentage:
alt.Chart(network_summary).mark_bar().encode(
    x=alt.X(
        "category:N",
        title="File type"
    ),
    y=alt.Y(
        "percentage_using_networking:Q",
        title="Files importing networking APIs (%)"
    ),
    tooltip=[
        "category:N",
        alt.Tooltip(
            "percentage_using_networking:Q",
            title="Percentage",
            format=".2f"
        ),
        "files_using_networking:Q",
        "files:Q"
    ]
).properties(
    title="Networking API Usage: Malware vs. Benign",
    width=500,
    height=350
)

alt.Chart(...)

In [17]:
"""Compare several API categories"""
api_categories = {
    "Networking": [
        "socket", "connect", "send", "recv",
        "internetopen", "httpsendrequest",
        "winhttp", "wsastartup"
    ],

    "File operations": [
        "createfile", "readfile", "writefile",
        "deletefile", "copyfile", "movefile"
    ],

    "Registry": [
        "regopenkey", "regsetvalue", "regqueryvalue",
        "regcreatekey", "regdeletekey"
    ],

    "Process and memory": [
        "createprocess", "openprocess",
        "virtualalloc", "virtualprotect",
        "writeprocessmemory", "createremotethread"
    ],

    "Cryptography": [
        "cryptencrypt", "cryptdecrypt",
        "cryptacquirecontext", "bcrypt",
        "certopenstore"
    ]
}

In [18]:
#Create category-level data:
category_rows = []

for category_name, terms in api_categories.items():
    counts = api_lists.apply(
        lambda names: count_matching_apis(names, terms)
    )

    for file_type, label_value in [
        ("Benign", 0),
        ("Malware", 1)
    ]:
        class_counts = counts[train_df["label"] == label_value]

        category_rows.append({
            "api_category": category_name,
            "file_type": file_type,
            "percentage_of_files": (
                (class_counts > 0).mean() * 100
            ),
            "average_count": class_counts.mean()
        })

api_category_df = pd.DataFrame(category_rows)

display(api_category_df.round(2))

,api_category,file_type,percentage_of_files,average_count
0,Networking,Benign,39.06,2.28
1,Networking,Malware,45.42,1.91
2,File operations,Benign,50.59,2.36
3,File operations,Malware,60.65,2.76
4,Registry,Benign,35.25,1.54
5,Registry,Malware,43.44,1.76
6,Process and memory,Benign,45.91,1.04
7,Process and memory,Malware,66.09,1.19
8,Cryptography,Benign,7.89,0.21
9,Cryptography,Malware,6.36,0.09


In [19]:
# Visualize:
alt.Chart(api_category_df).mark_bar().encode(
    x=alt.X(
        "api_category:N",
        title="API category"
    ),
    xOffset="file_type:N",
    y=alt.Y(
        "percentage_of_files:Q",
        title="Files importing category (%)"
    ),
    color=alt.Color(
        "file_type:N",
        title="File type"
    ),
    tooltip=[
        "api_category:N",
        "file_type:N",
        alt.Tooltip(
            "percentage_of_files:Q",
            title="Percentage",
            format=".2f"
        ),
        alt.Tooltip(
            "average_count:Q",
            title="Average API count",
            format=".2f"
        )
    ]
).properties(
    title="Imported API Categories: Malware vs. Benign",
    width=700,
    height=400
)

alt.Chart(...)

## Behavior TRAIN tag frequency

`win32_behavior_train_20pct.parquet` tags each sample with 0+ ClarAVy behavior labels (e.g. `backdoor`, `worm`, `downloader`). ClarAVy uses an empty string as a placeholder for "no confident tag," so that's filtered out before counting.

In [20]:


behavior_df = pd.read_parquet(DATA_DIR / "win32_behavior_train_20pct.parquet")

print("rows:", len(behavior_df))
display(behavior_df.head())

rows: 156357


,sha256,label,family,behavior,mbc,ttps
0,0015cbb011bbeb891b3ea505a6995d17395c4d4fd144b0...,1,berbew,[backdoor],[],[]
1,0020532d7497a685cf1d75668eef3184a0e683fd620090...,1,cosmu,[ransom],[],[]
2,00211af4c69b533a0663801dbb36617db6cc895ae3c6c8...,1,ctsinf,[],[],[]
3,0031339853d8f74ab0b22d4ea1501a43b8b2e28d972357...,1,koceg,[backdoor],[],[]
4,00328cb771644f2b8f05f76c4dda84179db343705d5141...,1,black,[worm],[],[]


In [21]:
#Flatten the behavior tag lists, dropping the empty-string "no tag" placeholder
behavior_tags = pd.Series(
    [tag for tags in behavior_df["behavior"] for tag in tags if tag]
)

tag_counts = (
    behavior_tags
    .value_counts()
    .rename_axis("behavior_tag")
    .reset_index(name="count")
)

print("unique behavior tags:", len(tag_counts))
display(tag_counts.head(10))

unique behavior tags: 88


,behavior_tag,count
0,backdoor,29187
1,worm,13219
2,virus,10465
3,downloader,8301
4,spyware,4204
5,dropper,3702
6,adware,2244
7,ransom,2201
8,selfmod,1972
9,packed,1699


In [22]:
#Horizontal bar chart of the top behavior tags by frequency
TOP_N = 15
top_tags = tag_counts.head(TOP_N)

alt.Chart(top_tags).mark_bar(
    color="#2a78d6",
    cornerRadiusEnd=4
).encode(
    x=alt.X("count:Q", title="Number of Samples"),
    y=alt.Y("behavior_tag:N", title="Behavior Tag", sort="-x"),
    tooltip=[
        alt.Tooltip("behavior_tag:N", title="Behavior tag"),
        alt.Tooltip("count:Q", title="Samples", format=",")
    ]
).properties(
    title=f"Top {TOP_N} Malware Behavior Tags",
    width=650,
    height=400
)

alt.Chart(...)

## Common co-occurring behavior tag pairs

Since `behavior` is multi-label, some samples carry 2+ tags (e.g. a backdoor that also spies on the user). This counts, per sample, every unordered pair of distinct real tags it carries, then ranks pairs by how often they co-occur.

In [23]:
from itertools import combinations
from collections import Counter

#Count each unordered pair of distinct real tags that co-occurs within a sample
pair_counts = Counter()
for tags in behavior_df["behavior"]:
    unique_tags = sorted(set(t for t in tags if t))
    for pair in combinations(unique_tags, 2):
        pair_counts[pair] += 1

pair_df = pd.DataFrame(
    [(f"{a} + {b}", count) for (a, b), count in pair_counts.items()],
    columns=["tag_pair", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

print("unique co-occurring pairs:", len(pair_df))
display(pair_df.head(10))

unique co-occurring pairs: 271


,tag_pair,count
0,backdoor + spyware,2782
1,adware + pua,835
2,backdoor + proxy,776
3,downloader + dropper,612
4,downloader + spyware,459
5,autorun + worm,396
6,backdoor + worm,340
7,virus + worm,301
8,downloader + injector,291
9,infector + virus,261


In [24]:
#Horizontal bar chart of the top co-occurring behavior tag pairs
TOP_N_PAIRS = 15
top_pairs = pair_df.head(TOP_N_PAIRS)

alt.Chart(top_pairs).mark_bar(
    color="#2a78d6",
    cornerRadiusEnd=4
).encode(
    x=alt.X("count:Q", title="Number of Samples"),
    y=alt.Y("tag_pair:N", title="Behavior Tag Pair", sort="-x"),
    tooltip=[
        alt.Tooltip("tag_pair:N", title="Tag pair"),
        alt.Tooltip("count:Q", title="Samples", format=",")
    ]
).properties(
    title=f"Top {TOP_N_PAIRS} Co-occurring Behavior Tag Pairs",
    width=650,
    height=400
)

alt.Chart(...)

VISUALIZATION FOR DETECTION TEST

Malware vs Benign distribution

# We examine the detection test-set class balance to confirm that benign and
# malware samples are represented fairly. A balanced test set makes evaluation
# metrics such as accuracy, precision, and recall easier to interpret.

In [28]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"

test_df = pd.read_parquet(
    DATA_DIR / "win32_test_detection.parquet"
)

print("Detection test shape:", test_df.shape)

Detection test shape: (359994, 5)


In [27]:
import altair as alt

alt.renderers.enable("default")
alt.data_transformers.enable("default")

DataTransformerRegistry.enable('default')

In [29]:
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

display(test_df.head())

Shape: (359994, 5)
Columns: ['sha256', 'label', 'general', 'strings', 'imports']


,sha256,label,general,strings,imports
0,00005da006489e8c288487dc2899c8afe341a610887ae7...,0,"{""size"": 123848, ""entropy"": 5.200856607961974,...","{""numstrings"": 912, ""avlength"": 11.49561403508...",{}
1,00014360be92b703d45d6db1c09f0916cdc580720fedc4...,1,"{""size"": 3000000, ""entropy"": 7.06598938246347,...","{""numstrings"": 22731, ""avlength"": 31.684175795...","{""oleaut32.dll"": [""SafeArrayPtrOfIndex"", ""Safe..."
2,00017862cd0c57bbabb30a3698f0181cdc5bb93c2c94db...,0,"{""size"": 884888, ""entropy"": 6.4442233179278325...","{""numstrings"": 5972, ""avlength"": 22.1989283322...","{""WININET.dll"": [""HttpSendRequestA"", ""HttpQuer..."
3,0001b778405e30b1bc7482907e479bc5eee255777c5c7e...,1,"{""size"": 135168, ""entropy"": 7.326132781964504,...","{""numstrings"": 632, ""avlength"": 7.704113924050...","{""MSVBVM60.DLL"": [""__vbaVarTstGt"", ""__vbaVarSu..."
4,00037ce21a123ae05be26f44b07fd45852c7100b70d6ec...,1,"{""size"": 238080, ""entropy"": 7.361566089709001,...","{""numstrings"": 1064, ""avlength"": 7.64567669172...","{""kernel32.dll"": [""LoadLibraryA"", ""GetProcAddr..."


In [30]:
test_label_counts = (
    test_df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

test_label_counts["category"] = test_label_counts["label"].map({
    0: "Benign",
    1: "Malware"
})

test_label_counts["percentage"] = (
    test_label_counts["count"]
    / test_label_counts["count"].sum()
    * 100
)

display(test_label_counts)

,label,count,category,percentage
0,1,180000,Malware,50.000833
1,0,179994,Benign,49.999167


In [31]:
import altair as alt

bars = (
    alt.Chart(test_label_counts)
    .mark_bar()
    .encode(
        x=alt.X(
            "category:N",
            title="File classification",
            sort=["Benign", "Malware"]
        ),
        y=alt.Y(
            "count:Q",
            title="Number of samples"
        ),
        color=alt.Color(
            "category:N",
            legend=None
        ),
        tooltip=[
            alt.Tooltip("category:N", title="Classification"),
            alt.Tooltip("count:Q", title="Samples", format=","),
            alt.Tooltip("percentage:Q", title="Percentage", format=".2f")
        ]
    )
)

labels = (
    alt.Chart(test_label_counts)
    .mark_text(
        dy=-10,
        fontSize=13
    )
    .encode(
        x=alt.X(
            "category:N",
            sort=["Benign", "Malware"]
        ),
        y="count:Q",
        text=alt.Text(
            "count:Q",
            format=","
        )
    )
)

test_class_balance_chart = (
    bars + labels
).properties(
    title="Detection Test Set: Malware vs. Benign Distribution",
    width=500,
    height=350
)

test_class_balance_chart

alt.LayerChart(...)

Train vs Test Behavior Distribution

In [37]:
import json
from pathlib import Path

import altair as alt
import pandas as pd


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    ROOT = CURRENT_DIR.parent
else:
    ROOT = CURRENT_DIR

DATA_DIR = ROOT / "win32_data"

behavior_train_df = pd.read_parquet(
    DATA_DIR / "win32_behavior_train_20pct.parquet"
)

behavior_test_df = pd.read_parquet(
    DATA_DIR / "win32_test_behavior.parquet"
)

print("Behavior train shape:", behavior_train_df.shape)
print("Behavior test shape:", behavior_test_df.shape)

print("\nTrain columns:", behavior_train_df.columns.tolist())
print("Test columns:", behavior_test_df.columns.tolist())

Behavior train shape: (156357, 6)
Behavior test shape: (180000, 6)

Train columns: ['sha256', 'label', 'family', 'behavior', 'mbc', 'ttps']
Test columns: ['sha256', 'label', 'family', 'behavior', 'mbc', 'ttps']


In [38]:
print("Train behavior example:")
print(behavior_train_df.iloc[0]["behavior"])

print("\nTest behavior example:")
print(behavior_test_df.iloc[0]["behavior"])

print("\nTrain behavior type:")
print(type(behavior_train_df.iloc[0]["behavior"]))

Train behavior example:
['backdoor']

Test behavior example:
['']

Train behavior type:
<class 'numpy.ndarray'>


In [43]:
import json
import numpy as np
import pandas as pd


def parse_json(value):
    if isinstance(value, (dict, list, np.ndarray)):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return value

    return value


def extract_behaviors(value):
    parsed_value = parse_json(value)

    if parsed_value is None:
        return []

    if isinstance(parsed_value, float) and pd.isna(parsed_value):
        return []

    # Convert NumPy arrays into regular Python lists.
    if isinstance(parsed_value, np.ndarray):
        parsed_value = parsed_value.tolist()

    if isinstance(parsed_value, list):
        behaviors = []

        for item in parsed_value:
            if isinstance(item, str):
                item = item.strip()

                if item:
                    behaviors.append(item)

            elif isinstance(item, dict):
                behavior_name = (
                    item.get("name")
                    or item.get("behavior")
                    or item.get("description")
                    or item.get("label")
                    or item.get("value")
                )

                if behavior_name:
                    behaviors.append(str(behavior_name))

        return behaviors

    if isinstance(parsed_value, str):
        parsed_value = parsed_value.strip()

        if parsed_value:
            return [parsed_value]

    if isinstance(parsed_value, dict):
        return [str(key) for key in parsed_value.keys()]

    return []

In [44]:
sample_behaviors = extract_behaviors(
    behavior_train_df.iloc[0]["behavior"]
)

print(sample_behaviors)

['backdoor']


In [45]:
behavior_train_df["extracted_behaviors"] = (
    behavior_train_df["behavior"]
    .apply(extract_behaviors)
)

display(
    behavior_train_df[
        ["behavior", "extracted_behaviors"]
    ].head(10)
)

,behavior,extracted_behaviors
0,[backdoor],[backdoor]
1,[ransom],[ransom]
2,[],[]
3,[backdoor],[backdoor]
4,[worm],[worm]
5,[],[]
6,[backdoor],[backdoor]
7,[worm],[worm]
8,[backdoor],[backdoor]
9,"[backdoor, spyware]","[backdoor, spyware]"


In [46]:
train_behavior_long = (
    behavior_train_df["extracted_behaviors"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

train_behavior_long = train_behavior_long[
    train_behavior_long != ""
]

print("Total extracted behavior records:", len(train_behavior_long))
print("Unique behaviors:", train_behavior_long.nunique())

print("\nMost common behaviors:")
print(train_behavior_long.value_counts().head(15))

Total extracted behavior records: 87539
Unique behaviors: 88

Most common behaviors:
extracted_behaviors
backdoor           29187
worm               13219
virus              10465
downloader          8301
spyware             4204
dropper             3702
adware              2244
ransom              2201
selfmod             1972
packed              1699
pua                 1403
injector            1245
passwordstealer      946
stealer              825
proxy                792
Name: count, dtype: int64


In [47]:
behavior_test_df["extracted_behaviors"] = (
    behavior_test_df["behavior"]
    .apply(extract_behaviors)
)

test_behavior_long = (
    behavior_test_df["extracted_behaviors"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

test_behavior_long = test_behavior_long[
    test_behavior_long != ""
]

print("Total test behavior records:", len(test_behavior_long))
print("Unique test behaviors:", test_behavior_long.nunique())

print("\nMost common test behaviors:")
print(test_behavior_long.value_counts().head(15))

Total test behavior records: 122337
Unique test behaviors: 93

Most common test behaviors:
extracted_behaviors
backdoor           49641
virus              16742
spyware            11603
worm                8488
downloader          8127
dropper             3334
adware              3157
proxy               2821
injector            1887
pua                 1832
packed              1716
ransom              1453
stealer             1398
autorun             1325
passwordstealer     1212
Name: count, dtype: int64


In [48]:
train_behavior_counts = (
    train_behavior_long
    .value_counts()
    .rename_axis("behavior")
    .reset_index(name="count")
)

train_behavior_counts["dataset"] = "Train"

test_behavior_counts = (
    test_behavior_long
    .value_counts()
    .rename_axis("behavior")
    .reset_index(name="count")
)

test_behavior_counts["dataset"] = "Test"

In [49]:
train_behavior_counts["percentage"] = (
    train_behavior_counts["count"]
    / train_behavior_counts["count"].sum()
    * 100
)

test_behavior_counts["percentage"] = (
    test_behavior_counts["count"]
    / test_behavior_counts["count"].sum()
    * 100
)

In [50]:
top_behaviors = (
    train_behavior_counts
    .head(15)["behavior"]
    .tolist()
)

print(top_behaviors)

['backdoor', 'worm', 'virus', 'downloader', 'spyware', 'dropper', 'adware', 'ransom', 'selfmod', 'packed', 'pua', 'injector', 'passwordstealer', 'stealer', 'proxy']


In [51]:
behavior_comparison_df = pd.concat(
    [
        train_behavior_counts[
            train_behavior_counts["behavior"].isin(top_behaviors)
        ],
        test_behavior_counts[
            test_behavior_counts["behavior"].isin(top_behaviors)
        ]
    ],
    ignore_index=True
)

In [52]:
complete_index = pd.MultiIndex.from_product(
    [
        top_behaviors,
        ["Train", "Test"]
    ],
    names=["behavior", "dataset"]
)

behavior_comparison_df = (
    behavior_comparison_df
    .set_index(["behavior", "dataset"])
    .reindex(complete_index, fill_value=0)
    .reset_index()
)

display(behavior_comparison_df)

,behavior,dataset,count,percentage
0,backdoor,Train,29187,33.341711
1,backdoor,Test,49641,40.577258
2,worm,Train,13219,15.100698
3,worm,Test,8488,6.938212
4,virus,Train,10465,11.954672
5,virus,Test,16742,13.685148
6,downloader,Train,8301,9.482631
7,downloader,Test,8127,6.643125
8,spyware,Train,4204,4.802431
9,spyware,Test,11603,9.484457


In [55]:
import altair as alt

behavior_distribution_chart = (
    alt.Chart(behavior_comparison_df)
    .mark_bar()
    .encode(
        y=alt.Y(
            "behavior:N",
            title="Behavior",
            sort=top_behaviors
        ),
        x=alt.X(
            "percentage:Q",
            title="Percentage of behavior records"
        ),
        yOffset=alt.YOffset(
            "dataset:N"
        ),
        color=alt.Color(
            "dataset:N",
            title="Dataset"
        ),
        tooltip=[
            alt.Tooltip(
                "behavior:N",
                title="Behavior"
            ),
            alt.Tooltip(
                "dataset:N",
                title="Dataset"
            ),
            alt.Tooltip(
                "count:Q",
                title="Occurrences",
                format=","
            ),
            alt.Tooltip(
                "percentage:Q",
                title="Percentage",
                format=".2f"
            )
        ]
    )
    .properties(
        title="Train vs. Test Behavior Distribution",
        width=650,
        height=500
    )
)

behavior_distribution_chart

alt.Chart(...)